# Fine-Tune SLM on Cocos2d-x JS/TS Training Data

This notebook fine-tunes a small language model on synthetic Cocos2d-x coding data.

**Requirements:** GPU runtime (T4 or better). Go to Runtime → Change runtime type → GPU.

## 1. Setup & Install Dependencies

In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl wandb

## 1b. Mount Google Drive for Persistent Storage

Checkpoints and adapters are saved to Drive so training survives Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Persistent paths on Google Drive
DRIVE_ROOT = "/content/drive/MyDrive/cocos2dx-finetune"
DRIVE_CHECKPOINTS = f"{DRIVE_ROOT}/checkpoints"
DRIVE_ADAPTER = f"{DRIVE_ROOT}/cocos2dx-lora-adapter"

import os
os.makedirs(DRIVE_CHECKPOINTS, exist_ok=True)
os.makedirs(DRIVE_ADAPTER, exist_ok=True)
print(f"Drive storage ready at {DRIVE_ROOT}")

## 2. Clone Repo & Load Data

In [ ]:
import os

# Clone the repo (change to your username if forked)
REPO = "nmnhut-it/fine-tune-cocoos
if not os.path.exists("fine-tune-cocoos"):
    !git clone https://github.com/{REPO}.git

os.chdir("fine-tune-cocoos")

In [ ]:
from datasets import load_dataset

train_ds = load_dataset("json", data_files="data/train.jsonl", split="train")
test_ds = load_dataset("json", data_files="data/test.jsonl", split="train")

print(f"Train: {len(train_ds)}, Test: {len(test_ds)}")
print(train_ds[0])

## 3. Choose Base Model

Pick an SLM. Good options:
- `microsoft/phi-2` (2.7B)
- `TinyLlama/TinyLlama-1.1B-Chat-v1.0` (1.1B)
- `Qwen/Qwen2.5-Coder-1.5B-Instruct` (1.5B)
- `google/codegemma-2b` (2B)

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"  # Change as needed

## 4. Load Model with QLoRA (4-bit)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# Use a dedicated pad token — NEVER reuse eos_token as pad_token
if tokenizer.pad_token is None or tokenizer.pad_token == tokenizer.eos_token:
    tokenizer.add_special_tokens({"pad_token": "<|pad|>"})
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.resize_token_embeddings(len(tokenizer))
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    use_dora=True,  # weight-decomposed LoRA — better for knowledge injection
)

model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable()
model.print_trainable_parameters()

## 5. Format Data as Alpaca Prompts (Two-Phase Tokenization)

Phase 1 (CPT): Full-text loss — model learns API vocabulary from entire text.
Phase 2 (SFT): Response-only loss — model learns to follow instruction format.

In [ ]:
ALPACA_TEMPLATE = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Response:
{output}"""

ALPACA_PROMPT = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Response:
"""

MAX_LENGTH = 1024
eos_id = tokenizer.eos_token_id

def tokenize_full_text(example):
    """Phase 1 (CPT): full-text loss on everything."""
    full = ALPACA_TEMPLATE.format(
        instruction=example["instruction"], output=example["output"],
    )
    tokens = tokenizer(full, truncation=True, max_length=MAX_LENGTH - 1, padding=False)
    tokens["input_ids"] = tokens["input_ids"] + [eos_id]
    tokens["attention_mask"] = tokens["attention_mask"] + [1]
    tokens["labels"] = list(tokens["input_ids"])
    return tokens

def tokenize_masked(example):
    """Phase 2 (SFT): loss only on response tokens."""
    prompt = ALPACA_PROMPT.format(instruction=example["instruction"])
    full = ALPACA_TEMPLATE.format(
        instruction=example["instruction"], output=example["output"],
    )
    tokens = tokenizer(full, truncation=True, max_length=MAX_LENGTH - 1, padding=False)
    tokens["input_ids"] = tokens["input_ids"] + [eos_id]
    tokens["attention_mask"] = tokens["attention_mask"] + [1]
    prompt_tokens = tokenizer(prompt, truncation=True, max_length=MAX_LENGTH)
    prompt_len = len(prompt_tokens["input_ids"])
    tokens["labels"] = [-100] * prompt_len + tokens["input_ids"][prompt_len:]
    return tokens

# Phase 1 datasets (full-text loss)
cpt_train = train_ds.map(tokenize_full_text, remove_columns=train_ds.column_names)
cpt_test = test_ds.map(tokenize_full_text, remove_columns=test_ds.column_names)

# Phase 2 datasets (response-only loss)
sft_train = train_ds.map(tokenize_masked, remove_columns=train_ds.column_names)
sft_test = test_ds.map(tokenize_masked, remove_columns=test_ds.column_names)

print(f"CPT data: {len(cpt_train)} train, {len(cpt_test)} test")
print(f"SFT data: {len(sft_train)} train, {len(sft_test)} test")

## 6. Two-Phase Training

In [ ]:
from dataclasses import dataclass
from typing import List, Dict, Any
from transformers import TrainingArguments, Trainer
import glob

@dataclass
class LabelPreservingCollator:
    """Pads batch to max length while preserving pre-computed labels (-100 masking)."""
    tokenizer: Any
    max_length: int = 1024

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        pad_id = self.tokenizer.pad_token_id
        batch = {}
        for key in ("input_ids", "attention_mask", "labels"):
            seqs = [f[key] for f in features]
            pad_val = -100 if key == "labels" else (0 if key == "attention_mask" else pad_id)
            max_len = min(max(len(s) for s in seqs), self.max_length)
            padded = []
            for s in seqs:
                diff = max_len - len(s)
                padded.append(s + [pad_val] * diff if diff > 0 else s[:max_len])
            batch[key] = torch.tensor(padded, dtype=torch.long)
        return batch

def make_trainer(train_data, test_data, output_dir, num_epochs, lr):
    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        learning_rate=lr,
        weight_decay=0.01,
        warmup_ratio=0.06,
        lr_scheduler_type="cosine",
        fp16=True,
        gradient_checkpointing=True,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        report_to="none",
        dataloader_pin_memory=False,
        neftune_noise_alpha=5.0,
    )
    return Trainer(
        model=model, args=args,
        train_dataset=train_data, eval_dataset=test_data,
        data_collator=LabelPreservingCollator(tokenizer),
    )

# === Phase 1: Continued Pre-Training (full-text loss) ===
print("=" * 60)
print("PHASE 1: Continued Pre-Training (full-text loss)")
print("=" * 60)
cpt_dir = f"{DRIVE_CHECKPOINTS}/phase1-cpt"
import os; os.makedirs(cpt_dir, exist_ok=True)

trainer = make_trainer(cpt_train, cpt_test, cpt_dir, num_epochs=3, lr=2e-4)
checkpoints = sorted(glob.glob(f"{cpt_dir}/checkpoint-*"))
resume = checkpoints[-1] if checkpoints else None
trainer.train(resume_from_checkpoint=resume)

# === Phase 2: Supervised Fine-Tuning (response-only loss) ===
print("=" * 60)
print("PHASE 2: Supervised Fine-Tuning (response-only loss)")
print("=" * 60)
sft_dir = f"{DRIVE_CHECKPOINTS}/phase2-sft"
os.makedirs(sft_dir, exist_ok=True)

trainer = make_trainer(sft_train, sft_test, sft_dir, num_epochs=3, lr=5e-5)
checkpoints = sorted(glob.glob(f"{sft_dir}/checkpoint-*"))
resume = checkpoints[-1] if checkpoints else None
trainer.train(resume_from_checkpoint=resume)

## 7. Save LoRA Adapter

In [ ]:
model.save_pretrained(DRIVE_ADAPTER)
tokenizer.save_pretrained(DRIVE_ADAPTER)
print(f"Adapter saved to {DRIVE_ADAPTER}")
print("Files:", os.listdir(DRIVE_ADAPTER))

## 8. Test Inference

In [ ]:
def generate(instruction, max_new_tokens=512):
    prompt = f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Response:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

# Test with examples from the test set
for i in [0, 25, 50, 75]:
    example = test_ds[i]
    print(f"\n{'='*60}")
    print(f"INSTRUCTION: {example['instruction'][:200]}...")
    print(f"\nGENERATED:\n{generate(example['instruction'])}")
    print(f"\nEXPECTED:\n{example['output'][:300]}...")

## 9. Evaluate on Test Set

In [ ]:
eval_results = trainer.evaluate()
print(f"Test Loss: {eval_results['eval_loss']:.4f}")
print(f"Test Perplexity: {2**eval_results['eval_loss']:.2f}")

## 10. (Optional) Push Adapter to Hugging Face Hub

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()
# model.push_to_hub("your-username/cocos2dx-coder-lora")
# tokenizer.push_to_hub("your-username/cocos2dx-coder-lora")